# Getting Started with cntmosaic
In this guide, we will walk through installing and using the `cntmosaic` package to process data, run simulation and visualize results.

Current model supports fine integer age for participants, contacts and population. 

## Imports

In [1]:
from cntmosaic.dataloader import CoordToColumns, DataLoader
from cntmosaic.models import BRCfine

import pandas as pd
import numpy as np

import jax
from numpyro import distributions as dist
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_value

##  Load Example Data

We will use the famous POLYMOD data as an example, you can get the csv files from https://zenodo.org/records/3746312#.Xo85CcgzZPY

Here we will show an interface similar to Socialmixr, a dataframe for participant, a dataframe for contacts and optionally, a dataframe for population.

In [2]:
cnt = pd.read_csv('2008_Mossong_POLYMOD_contact_common.csv')
part = pd.read_csv('2008_Mossong_POLYMOD_participant_common.csv')

pop = pd.DataFrame({
	'age': np.arange(0, 85, dtype=int),
	'size': np.ones(85, dtype=int),
})

The model requires columns containing the participant age, the contact age, the id variable that connects the two dataframe. Other optional input include the contact count `y`. By default, each row represents one contact. Population characteristics can also be set, which will be demonstrated later.

`CoordToColumn` class can map the required columns to the model's need without changing the column name. Simply pass in the corresponding column name as shown below:

In [3]:
mapper = CoordToColumns(age_part='part_age', 
                        age_cnt='cnt_age_exact',
                        age_pop='age',
                        size_pop='size',
                        id_var='part_id')

Afterwards, we can call the dataloader. It will merge by `id_var`. Use `data.raw_df` to inspect merged dataframe. Direct changes to `raw_df` are allowed. Then, use the $load$ function to prepare the dataloader to model use.

In [4]:
dataloader = DataLoader(part, cnt, pop, mapper)

/Users/shozendan/Imperial/0_Research/cntmosaic/cntmosaic/dataloader/_dataloader.py:171: UserWarning: Maximum age in sample (99.0) does not match that in the population information (84). Adjusting sample maximum age to match population.
  warnings.warn(f"Maximum age in sample ({sample_max_age}) does not match that in the population information ({pop_max_age}). Adjusting sample maximum age to match population.", UserWarning)


## Call the Model

In [5]:
from cntmosaic.models.priors import PenalisedTensorSpline2D

priors = {
	'rate': PenalisedTensorSpline2D(
		grid_type='age-age',
		type='global',
		symmetric=True
	)
}

Both interface will generate a loaded dataloader object. We can pass this dataloader object to our model.

In [6]:
model = BRCfine(dataloader, priors, 'negbin')

Default parameters can be modified easily. `M` indicates the number of basis functions used to approximate the Gaussian Process, `C` is the boundary factor. We recommend keeping them unchanged. `grid-type` supports 'age-age' and 'diff-age'. If you suspect the contact dynamics have dependence on the age difference between participants and contacts, 'diff-age' would be more suitable. Additional offsets to contact intensity is possible via `offset`. It needs to be of the same dimension as the age grid.

The priors are numpryo distributions. `beta0` is the baseline parameter; `alpha` is the magnitude and `rho` is the lengthscales for HSGP. Please note `rho` is 2-D. Example below shows how to modify them.

We can generate samples from these priors.

Now we can run MCMC simulation.

In [8]:
prng_key = jax.random.PRNGKey(1)
init_values = {'baseline': -model.log_P.mean()}
guide = AutoNormal(model.model, init_loc_fn=init_to_value(values=init_values))
model.run_inference_svi(prng_key, guide)

100%|██████████| 5000/5000 [00:26<00:00, 190.92it/s, init loss: 84329.7969, avg. loss [4751-5000]: 18422.8210]


In [9]:
from cntmosaic.analysis import ModelSummariserSVI, ModelVisualiser

summariser = ModelSummariserSVI(model)

In [14]:
model.posterior_predictive_svi(prng_key, guide)

{'log_cint': Array([[[-3.2382464, -3.3692055, -3.5560865, ..., -6.5294065,
          -6.50167  , -6.494089 ],
         [-3.3692055, -3.1168842, -3.2777805, ..., -6.575058 ,
          -6.521612 , -6.5138016],
         [-3.5560865, -3.2777805, -2.9320614, ..., -6.5938015,
          -6.5452065, -6.554343 ],
         ...,
         [-6.5294065, -6.575058 , -6.5938015, ..., -3.7503514,
          -3.8351443, -3.941834 ],
         [-6.50167  , -6.521612 , -6.5452065, ..., -3.8351443,
          -3.8788466, -3.979824 ],
         [-6.494089 , -6.5138016, -6.554343 , ..., -3.941834 ,
          -3.979824 , -3.990085 ]],
 
        [[-3.037994 , -3.173028 , -3.3734446, ..., -6.807252 ,
          -6.776595 , -6.7604666],
         [-3.173028 , -3.0188458, -3.1500711, ..., -6.751191 ,
          -6.763361 , -6.792018 ],
         [-3.3734446, -3.1500711, -2.9775925, ..., -6.751583 ,
          -6.7792425, -6.8239384],
         ...,
         [-6.807252 , -6.751191 , -6.751583 , ..., -3.9847703,
          -4

It is recommended to save the model after MCMC is done.

In [12]:
import pickle
with open('diff-age.pkl', 'wb') as f:
    pickle.dump(model, f)


## Visualize Model Outputs

We can call the summariser and visualiser to see the posterior contact intensity and contact rate.

In [ ]:
from cntmosaic.analysis import ModelSummariserMCMC, ModelVisualiser
MSM = ModelSummariserMCMC(model)
visual = ModelVisualiser(MSM)

In [ ]:
visual.plot_cint()['general']

In [ ]:
visual.plot_rate()

## 📚 Further Exploration
We can also extract posterior draws for each of our variables in the `post` attribute. Posterior contact intensity is stored in `post_cint`. 

In [ ]:
MSM.post.keys()

In [ ]:
MSM.post_cint['general'].shape